In [6]:
# print('llm 调用tools 让llm 和外界交流')
# llm 调用tools 让llm 和外界交流
# # 发起请求的服务

# !pip install requests
# !pip install openai
import requests
# python 类型约定，
# js 弱类型语言 
# -> str 返回值的类型
def get_weather(location: str) -> str:
    url = "https://api.seniverse.com/v3/weather/now.json"
    params = {
        "key": "SaVSOt7sYbwpka9iv",
        "location": location,
        "language": "zh-Hans"
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        # print(data)
        if "results" in data:
            r = data["results"][0]
            city = r["location"]["name"]
            # 当前天气对象
            now = r["now"]
            text = now["text"]
            temp = now["temperature"]
            #python 擅长机器学习 和字符处理
            return f"{city}当前天气：{text}, 气温 {temp}度"
        else:
            return "查询失败"
    except Exception as e:
        return  f"异常：{e}"
    
# print(get_weather("抚州"))
# {'results': [{'location': {'id': 'WSFXR95RZD21', 'name': '抚州', 'country': 'CN', 'path': '抚州,抚州,江西,中国', 'timezone': 'Asia/Shanghai', 'timezone_offset': '+08:00'}, 'now': {'text': '阴', 'code': '9', 'temperature': '8'}, 'last_update': '2025-11-18T20:37:05+08:00'}]}
# 抚州当前天气：阴, 气温 8度
from openai import OpenAI
client=OpenAI(
    api_key='sk-f0085aa7075f45ce946b9515da8c4cb2',
    base_url='https://api.deepseek.com'
)
# openai 接口定义
tools=[
{
    "type":"function",
    "function":{
        "name":"get_weather",
        "description":"获取制定城市的当前天气",
        "parameters":{
            # 定义函数接受的参数结构
            "type":"object",
            # 定义了具体的参数属性
            "properties":{
                    "location":{
                        "type":"string",
                        # 北京天气怎么样？ 提取出来 北京
                        "description":"城市名称，如‘北京’"
                    }
            },
            "required":["location"]
        }
    }
}
]

import json
messages=[{"role":"user","content":"北京天气怎么样"}]
response=client.chat.completions.create(
    model='deepseek-reasoner',
    messages=messages,
    tools=tools,
    tool_choice="auto",
    # 生成的随机度
    temperature=0.1
)
response_message=response.choices[0].message
print(response_message)
messages.append(response_message)

if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        print(tool_call.function)
        function_name=tool_call.function.name
        # json字符串 变成json对象
        function_args=json.loads(tool_call.function.arguments)
        if function_name=="get_weather":
            function_response=get_weather(function_args["location"])
        else:
            function_response="未知工具"
        
        messages.append({
            "tool_call_id":tool_call.id,
            "role":"tool",
            "name":function_name,
            "content":function_response
        })
else:
       print( response_message.content)

final_response=client.chat.completions.create(
    model="deepseek-reasoner",
    messages=messages,
    temperature=0.3
)
# print(final_response.choices[0].message.content)

ChatCompletionMessage(content='我来帮您查询北京的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_7rmllIXLhYksPMfp9o6oLcJl', function=Function(arguments='{"location": "北京"}', name='get_weather'), type='function', index=0)])
Function(arguments='{"location": "北京"}', name='get_weather')
